# 〰️ Beyond Linearity
**ISLP Chapter 7 · Pattern Recognition for the Rest of Us**

> Linear regression assumes a straight-line relationship. But many real-world relationships are curved. This notebook covers flexible extensions: polynomial regression, splines, and Generalized Additive Models — all interpretable ways to model nonlinearity.

### What you'll learn
- Polynomial regression: fitting curves with OLS
- Step functions: piecewise constants
- Regression splines: smooth curves with knots
- Natural cubic splines
- GAMs: additive models with one smooth term per feature

### Dataset: Wage (ISLP) + Auto
### Time: ~55 minutes

In [ ]:
# Overview: income vs age relationship
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(wage['age'], wage['wage'], alpha=0.15, color='#888', s=10)
ax.set_xlabel('Age'); ax.set_ylabel('Wage ($)')
ax.set_title('Wage vs Age — clearly nonlinear, needs more than a straight line')
plt.tight_layout(); plt.show()

## 📐 Part 1 — Polynomial Regression

Extend linear regression with polynomial terms:
```
Y = β₀ + β₁X + β₂X² + β₃X³ + ... + βdXd + ε
```
This is still linear regression (linear in the β coefficients) — OLS works perfectly. The trick is just creating the polynomial features from X.

In [ ]:
# Polynomial regression: compare degrees
X_age = wage['age'].values.reshape(-1,1)
y_wage = wage['wage'].values
age_range = np.linspace(wage['age'].min(), wage['age'].max(), 200).reshape(-1,1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
degrees = [1, 2, 4, 10]
for ax, d in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#1e5fa8', lw=2.5)
    ax.set_title(f'Degree {d}\n10-fold CV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Polynomial Regression: Increasing Degree', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Degree 4 captures the curve. Degree 10 wiggles — overfitting at the extremes')

## 🪢 Part 2 — Splines: Smooth Piecewise Polynomials

Polynomials are global — a wiggle at one end affects predictions everywhere. **Splines** are piecewise polynomials that join smoothly at **knots**.

A **regression spline** with K knots has K+d+1 basis functions (d = degree). The result: flexibility where the data is dense, smoothness everywhere.

**Natural cubic splines** add the constraint that the function is linear beyond the boundary knots — preventing wild behavior at the extremes.

In [ ]:
# Compare: linear, polynomial, spline
from sklearn.preprocessing import SplineTransformer

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_spline = [
    ('Degree-4 Poly', Pipeline([('poly',PolynomialFeatures(4)),('lr',LinearRegression())])),
    ('Spline (4 knots)', Pipeline([('spline',SplineTransformer(n_knots=4, degree=3)),('lr',LinearRegression())])),
    ('Spline (8 knots)', Pipeline([('spline',SplineTransformer(n_knots=8, degree=3)),('lr',LinearRegression())])),
]

for ax, (title, pipe) in zip(axes, models_spline):
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#e85d20', lw=2.5)
    ax.set_title(f'{title}\nCV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Splines vs Polynomial', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## ➕ Part 3 — Generalized Additive Models (GAMs)

GAMs extend linear models to allow nonlinear relationships:
```
Y = β₀ + f₁(X₁) + f₂(X₂) + ... + fₚ(Xₚ) + ε
```
Each fⱼ is a smooth function estimated from the data. The **additive** structure means:
- Each feature contributes independently
- Interpretable: plot each fⱼ to see its effect
- Much more flexible than linear but more interpretable than neural networks

In [ ]:
!pip install pygam -q
from pygam import LinearGAM, s, f, l

# GAM: wage ~ s(age) + s(year) + education
wage_enc = pd.get_dummies(wage[['age','year','education','wage']], drop_first=True, dtype=float)
feature_cols = [c for c in wage_enc.columns if c != 'wage']
X_gam = wage_enc[feature_cols].values
y_gam = wage_enc['wage'].values

gam = LinearGAM(s(0) + s(1) + l(2) + l(3) + l(4) + l(5)).fit(X_gam, y_gam)
print(f'GAM R²: {gam.statistics_["pseudo_r2"]["McFadden"]:.4f}')
gam.summary()

In [ ]:
# Plot GAM smooth terms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, term_idx, label in zip(axes, [0, 1], ['age', 'year']):
    XX = gam.generate_X_grid(term=term_idx)
    pdep, confi = gam.partial_dependence(term=term_idx, width=0.95)
    ax.plot(XX[:,term_idx], pdep, color='#1e5fa8', lw=2.5)
    ax.fill_between(XX[:,term_idx], confi[:,0], confi[:,1], alpha=0.2, color='#1e5fa8')
    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel(label); ax.set_ylabel(f'Effect on wage')
    ax.set_title(f'GAM: smooth term for {label}')
plt.suptitle('GAM Partial Effects — Age and Year', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Wage rises steeply with age until ~40, then levels off')
print('   Year shows a gradual upward trend — wages rising over time')

In [ ]:
# @title 📝 Quiz — Beyond Linearity
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is a regression spline and how does it differ from a polynomial?
# @markdown **Q2:** What are knots in the context of splines?
# @markdown **Q3:** What is the additive assumption in GAMs?
# @markdown **Q4:** How do you choose the number/location of knots?
# @markdown **Q5:** Name one advantage of splines over high-degree polynomials

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Beyond Linearity"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
